# The Feature Store Approach

To organize the variables used in the model, a **Feature Store** approach was adopted. The goal is to separate the **feature engineering** from the modeling stage, allowing the same variables to be used consistently during both training and prediction.

Feature construction:

The variables are created using **SQL queries**, which are stored in separate *.sql* files. This organization helps keep the logic for each feature group isolated, making it easier to maintain and reuse them.

The structure is as follows, for example:

```text
feature_store/
├── fs_temporal.sql
├── fs_store.sql
├── fs_sales.sql
└── fs_customers.sql
```

The queries are parameterized by the reference period. This way, the same query can be run for different months, generating features for each corresponding DATA_REF.

Ingesting into the Feature Store:

The execution of queries is centralized in an **ingestion notebook**. This notebook reads each *.sql* file, executes the query for the defined periods, and saves the results in their respective tables in the Feature Store.

The flow can be represented as:

```text
Raw data
     ↓
SQL queries
     ↓
Feature engineering
     ↓
Ingestion notebook
     ↓
Feature Store
     ↓
Training Set / Prediction
```

On the first run, if the table does not yet exist, it is created with:

* *STORE*
* *DATA_REF*

as the feature keys;

* *DATA_REF* as the partitioning column.

On subsequent runs, new periods are added using **merge**, allowing the Feature Store to be updated without recreating the entire table.

> **Important:** Since this is a temporal problem, historical features must be constructed using only information available up to the respective *DATA_REF*. This prevents future information from being used during training and reduces the risk of data leakage.



The features were divided into different groups according to their origin and purpose:

* **Temporal**: Features related to store operations, promotions, holidays, and temporal events observed up to the reference date;

* **Store**: Features related to the structural characteristics of the store, competition, and participation in promotional programs;

* **Sales**: Features related to the sales history, behavior, seasonality, and trends of the store;

* **Customers**: Features related to the customer behavior and the volume of customers at the store;


## Feature Store Temporal


**Key:** STORE_ID + REF_DATE

Features related to store operations, promotions, holidays, and temporal events observed up to the reference date.


**Features**

* **Days with store open:** number of days the store was open in the last 7, 14, 28, 42, 56, and 84 days.

* **Days with store closed:** number of days the store was closed in the last 7, 14, 28, 42, 56, and 84 days.

* **Operational rate:** proportion of days the store was open in the last 7, 14, 28, 42, 56, and 84 days.

* **Days with promotion:** number of days the store had a promotion in the last 7, 14, 28, 42, 56, and 84 days.

* **Days without promotion:** number of days the store did not have a promotion in the last 7, 14, 28, 42, 56, and 84 days.

* **Promotion rate:** proportion of days the store had a promotion in the last 7, 14, 28, 42, 56, and 84 days.

* **Days with state holiday:** number of days with state holidays in the last 7, 14, 28, 42, 56, and 84 days.

* **State holiday rate:** proportion of days with state holidays in the last 7, 14, 28, 42, 56, and 84 days.

* **Days with school holiday:** number of days with school holidays in the last 7, 14, 28, 42, 56, and 84 days.

* **School holiday rate:** proportion of days with school holidays in the last 7, 14, 28, 42, 56, and 84 days.

* **Time since competition started:** number of days elapsed between the competition opening date and the reference date.

* **Time since Promo2 started:** number of days elapsed between the Promo2 start date and the reference date.


## Feature Store Store


**Key:** STORE_ID + REF_DATE

Features related to the structural characteristics of the store, competition, and participation in promotional programs.



**Features**

* **Active competition:** Indicator showing whether there was an active competitor for the store on the reference date.

* **Active Promo2:** Indicator showing whether the store was participating in an active Promo2 campaign on the reference date.

* **Store type:** Classification of the store according to its type.

* **Assortment type:** Classification of the product assortment offered by the store.

* **Distance to competition:** Distance, in meters, from the store to the nearest competitor.


## Feature Store Sales

**Key:** STORE_ID + REF_DATE

Features related to the store's sales history, behavior, seasonality, and trends, considering only the information available up to the reference date.

**Features**

* **Sales sum:** total value of sales over the last 7, 14, 28, 42, 56, and 84 days.

* **Average sales:** daily average of sales over the last 7, 14, 28, 42, 56, and 84 days.

* **Minimum sales:** lowest daily sales value in the last 7, 14, 28, 42, 56, and 84 days.

* **Maximum sales:** highest daily sales value in the last 7, 14, 28, 42, 56, and 84 days.

* **Sales standard deviation:** measure of daily sales variability over the last 7, 14, 28, 42, 56, and 84 days.

* **Median sales:** the median of daily sales over the last 7, 14, 28, 42, 56, and 84 days.

* **Sales sum during promotion:** total sales on days when the store was in promotion over the last 7, 14, 28, 42, 56, and 84 days.

* **Average sales during promotion:** average daily sales on days when the store was in promotion over the last 7, 14, 28, 42, 56, and 84 days.

* **Sales sum without promotion:** total sales on days when the store was not in promotion over the last 7, 14, 28, 42, 56, and 84 days.

* **Average sales without promotion:** average daily sales on days without promotion over the last 7, 14, 28, 42, 56, and 84 days.

* **Sales lift during promotions:** percentage difference between the average sales during promotion periods and the average sales during periods without promotions for the last 7, 14, 28, 42, 56, and 84 days.

* **Average sales for the same period in the previous year:** average sales in the 42 equivalent days of the previous year.

* **Growth compared to the previous year:** percent change in average sales for the last 42 days compared to the average sales for the equivalent period in the previous year.

* **Sales sum by weekday:** total sales for each weekday considering the last 4, 8, and 12 occurrences of that specific weekday.

* **Average sales by weekday:** average sales for each weekday considering the last 4, 8, and 12 occurrences of that specific weekday.

* **Sales standard deviation by weekday:** variability of sales for each weekday considering the last 4, 8, and 12 occurrences of that specific weekday.

* **Sales per customer:** ratio between total sales and total customers for the last 7, 14, 28, 42, 56, and 84 days.

* **Daily sales history:** individual sales values observed during the last 84 days before the reference date.

* **7D vs. 28D sales growth:** percent growth in average sales over the last 7 days compared to the last 28 days.

* **14D vs. 28D sales growth:** percent growth in average sales over the last 14 days compared to the last 28 days.

* **28D vs. 56D sales growth:** percent growth in average sales over the last 28 days compared to the last 56 days.

* **42D vs. 84D sales growth:** percent growth in average sales over the last 42 days compared to the last 84 days.


## Feature Store Customer



**Key:** STORE_ID + REF_DATE

Features related to customer behavior and volume at the store, considering the historical data observed up to the reference date.


**Features**

* **Total customers:** total number of customers in the last 7, 14, 28, 42, 56, and 84 days.

* **Average customers:** average daily number of customers in the last 7, 14, 28, 42, 56, and 84 days.

* **Minimum customers:** lowest daily number of customers in the last 7, 14, 28, 42, 56, and 84 days.

* **Maximum customers:** highest daily number of customers in the last 7, 14, 28, 42, 56, and 84 days.

* **Standard deviation of customers:** daily variability of customer counts in the last 7, 14, 28, 42, 56, and 84 days.

* **Median of customers:** median of the daily customer counts in the last 7, 14, 28, 42, 56, and 84 days.

* **Total customers during promotion:** total number of customers on days when the store was in promotion in the last 7, 14, 28, 42, 56, and 84 days.

* **Average customers during promotion:** average daily number of customers on days when the store was in promotion in the last 7, 14, 28, 42, 56, and 84 days.

* **Total customers without promotion:** total number of customers on days when the store was not in promotion in the last 7, 14, 28, 42, 56, and 84 days.

* **Average customers without promotion:** average daily number of customers on days when the store was not in promotion in the last 7, 14, 28, 42, 56, and 84 days.

* **Customer growth 7D vs. 28D:** percentage growth of the average number of customers in the last 7 days compared to the last 28 days.

* **Customer growth 14D vs. 28D:** percentage growth of the average number of customers in the last 14 days compared to the last 28 days.

* **Customer growth 28D vs. 56D:** percentage growth of the average number of customers in the last 28 days compared to the last 56 days.

* **Customer growth 42D vs. 84D:** percentage growth of the average number of customers in the last 42 days compared to the last 84 days.


## Note

> These are the initially proposed features. The final definition may change during exploration and modeling, depending on data availability, predictive relevance, and the need to avoid *data leakage*.